# Week 4, Day 5: Orthogonality, projection and representational similarity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w004/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Is a set orthonormal?

Write `is_orthonormal(Q)` that returns True when the columns of `Q` are orthonormal, by checking that `Q.T @ Q` is close to the identity matrix (`np.allclose` and `np.eye`).

In [ ]:
import numpy as np

def is_orthonormal(Q):
    ...

print(is_orthonormal(np.eye(2)), is_orthonormal(np.array([[1.0, 1.0], [0.0, 1.0]])))

In [ ]:
import numpy as np
def test_cases():
    "is_orthonormal gives the right answers"
    assert is_orthonormal(np.eye(3))
    c, s = np.cos(0.3), np.sin(0.3)
    assert is_orthonormal(np.array([[c, -s], [s, c]]))
    assert not is_orthonormal(np.array([[1.0, 1.0], [0.0, 1.0]]))
    assert not is_orthonormal(np.array([[2.0, 0.0], [0.0, 2.0]]))

test_cases()
print("All checks passed")

### Exercise: Gram-Schmidt in NumPy

Write `gram_schmidt(V)` that takes a matrix with independent columns and returns a matrix `Q` with orthonormal columns spanning the same space, following the derivation.

In [ ]:
def gram_schmidt(V):
    Q = np.zeros_like(V, dtype=float)
    for k in range(V.shape[1]):
        u = V[:, k].astype(float)
        for j in range(k):
            ...
        Q[:, k] = ...
    return Q

V = np.array([[3.0, 1.0], [4.0, 2.0]])
Q = gram_schmidt(V)
print(Q); print(np.round(Q.T @ Q, 10))

In [ ]:
import numpy as np
def test_orthonormal():
    "the output columns are orthonormal"
    V = np.random.default_rng(0).normal(size=(5, 3))
    Q = gram_schmidt(V)
    assert Q.shape == (5, 3) and np.allclose(Q.T @ Q, np.eye(3))

def test_same_span():
    "the output spans the same space as the input"
    V = np.random.default_rng(1).normal(size=(4, 2))
    Q = gram_schmidt(V)
    coeffs = np.linalg.lstsq(Q, V, rcond=None)[0]
    assert np.allclose(Q @ coeffs, V)

def test_first_direction():
    "the first output column points along the first input column"
    V = np.array([[3.0, 1.0], [4.0, 2.0]])
    assert np.allclose(gram_schmidt(V)[:, 0], [0.6, 0.8])

test_orthonormal()
test_same_span()
test_first_direction()
print("All checks passed")

### Exercise: Remove a shared arousal signal

`X` has 300 trials by 40 neurons. Every neuron gets the same arousal fluctuation (along `u`, the normalised all-ones direction) plus a task signal along `task_dir`, which is orthogonal to `u`. Compute `cleaned` by removing the component along `u` from every trial, with one matrix expression and no loop.

In [ ]:
rng = np.random.default_rng(3)
n = 40
u = np.ones(n) / np.sqrt(n)                         # arousal: every neuron up together
task_dir = rng.normal(size=n); task_dir -= (task_dir @ u) * u; task_dir /= np.linalg.norm(task_dir)
arousal = rng.normal(0, 8, 300)
task = rng.normal(0, 1, 300)
X = 10 + np.outer(arousal, u) * np.sqrt(n) / 4 + np.outer(task, task_dir) * 3 + rng.normal(0, 1, (300, n))

cleaned = ...
print("variance along u before:", np.var(X @ u).round(1), " after:", np.var(cleaned @ u).round(6))

In [ ]:
import numpy as np
def test_orthogonal():
    "cleaned activity has no component along u"
    assert np.allclose(cleaned @ u, 0, atol=1e-9)

def test_rest_kept():
    "everything orthogonal to u is unchanged"
    P = X - np.outer(X @ u, u)
    assert np.allclose(cleaned, P)

def test_task_survives():
    "the task signal survives the cleaning"
    r = np.corrcoef(cleaned @ task_dir, task)[0, 1]
    assert r > 0.8, f"correlation {r:.2f}"

test_orthogonal()
test_rest_kept()
test_task_survives()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(5)
oris = np.arange(0, 180, 15.0)                              # 12 stimuli
prefs = np.linspace(0, 180, 60, endpoint=False)            # 60 "V1" neurons
def tuning(theta, pref, kappa=2.0):
    return 3 + 25 * np.exp(kappa * (np.cos(2 * np.radians(theta - pref)) - 1))
brain = np.array([[tuning(o, p) for p in prefs] for o in oris]) + rng.normal(0, 2, (12, 60))   # (stimuli, neurons)
print("brain responses:", brain.shape)

### Exercise: Build and compare RDMs

Write `rdm(R)` for a (stimuli, units) array: the matrix of $1 - r$ between every pair of rows (use `np.corrcoef`, which correlates rows). Write `compare(D1, D2)`: the correlation between the two matrices' entries above the diagonal (`np.triu_indices`). Compare the brain with a model that has the right tuning and with one whose stimulus labels are shuffled.

In [ ]:
def rdm(R):
    ...

def compare(D1, D2):
    i, j = np.triu_indices(D1.shape[0], k=1)
    ...

model_good = np.array([[tuning(o, p, kappa=1.2) for p in np.linspace(0, 180, 25, endpoint=False)] for o in oris])   # a different number of units
model_bad = model_good[rng.permutation(12)]                                                                     # same units, stimuli scrambled

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (name, R) in zip(axes, [("brain", brain), ("good model", model_good), ("shuffled model", model_bad)]):
    ax.imshow(rdm(R), cmap="viridis"); ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("brain vs good model:", round(compare(rdm(brain), rdm(model_good)), 2), "  brain vs shuffled:", round(compare(rdm(brain), rdm(model_bad)), 2))

In [ ]:
import numpy as np
def test_rdm_shape():
    "rdm returns a symmetric matrix with zeros on the diagonal"
    D = rdm(brain)
    assert D.shape == (12, 12) and np.allclose(D, D.T) and np.allclose(np.diag(D), 0)

def test_rdm_values():
    "entries are one minus the correlation of response patterns"
    D = rdm(brain)
    assert abs(D[0, 3] - (1 - np.corrcoef(brain[0], brain[3])[0, 1])) < 1e-9

def test_compare():
    "the matching model correlates better with the brain than the shuffled model"
    assert compare(rdm(brain), rdm(model_good)) > compare(rdm(brain), rdm(model_bad)) + 0.3

test_rdm_shape()
test_rdm_values()
test_compare()
print("All checks passed")